In [2]:
!pip install catboost -q


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [5]:
df = sns.load_dataset('titanic')
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


# Preprocessing of the data

In [9]:
# impute missing values using knn imputers in fare, age, embark_town, and embarked
from sklearn.impute import KNNImputer
from sklearn.impute import SimpleImputer

# Numerical columns
knn_imputer = KNNImputer(n_neighbors=5)

df['age'] = knn_imputer.fit_transform(df[['age']])
df['fare'] = knn_imputer.fit_transform(df[['fare']])

# Categorical columns
cat_imputer = SimpleImputer(strategy='most_frequent')

df['embark_town'] = cat_imputer.fit_transform(df[['embark_town']]).ravel()

df['embarked'] = cat_imputer.fit_transform(df[['embarked']]).ravel()

In [11]:
df.drop('deck', axis=1, inplace=True)

In [12]:
df.isnull().sum()

survived       0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
class          0
who            0
adult_male     0
embark_town    0
alive          0
alone          0
dtype: int64

In [13]:
# convert each categorical columns to category
categorical_cols = df.select_dtypes(include=['object', 'category']).columns

In [14]:
# add this as a new column in the dataframe
df[categorical_cols] = df[categorical_cols].astype('category')

In [15]:
# split data X and y
X = df.drop('survived', axis=1)
y = df['survived']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [21]:
# Run the Catboost classifier
from catboost import CatBoostClassifier

# categorical columns
categorical_cols = X_train.select_dtypes(
    include=['object', 'category']
).columns.tolist()

print(categorical_cols)

# model
model = CatBoostClassifier(
    iterations=100,
    learning_rate=0.1,
    depth=3,
    loss_function='Logloss',
    eval_metric='Accuracy',
    random_state=42,
    verbose=False
)

# train
model.fit(X_train, y_train, cat_features=categorical_cols)

['sex', 'embarked', 'class', 'who', 'embark_town', 'alive']


CatBoostClassifier(depth=3, eval_metric='Accuracy', iterations=100, learning_rate=0.1, loss_function='Logloss', random_state=42, verbose=False)

In [22]:
# Prediction
y_pred = model.predict(X_test)

In [25]:
# Evaluate the model
print('Accuracy : \n',accuracy_score(y_test,y_pred))
print('Confusion Matrix :\n ', confusion_matrix(y_test,y_pred))
print('Classification report : \n', classification_report(y_test,y_pred))

Accuracy : 
 1.0
Confusion Matrix :
  [[105   0]
 [  0  74]]
Classification report : 
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       105
           1       1.00      1.00      1.00        74

    accuracy                           1.00       179
   macro avg       1.00      1.00      1.00       179
weighted avg       1.00      1.00      1.00       179

